<a href="https://colab.research.google.com/github/hbisgin/DeepLearning/blob/main/DL_15_RNN_Name_LangExample_w_Embedding.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!wget https://raw.githubusercontent.com/EdwardRaff/Inside-Deep-Learning/refs/heads/main/idlmam.py

--2026-03-29 21:08:26--  https://raw.githubusercontent.com/EdwardRaff/Inside-Deep-Learning/refs/heads/main/idlmam.py
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 2606:50c0:8002::154, 2606:50c0:8003::154, 2606:50c0:8001::154, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|2606:50c0:8002::154|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 22064 (22K) [text/plain]
Saving to: ‘idlmam.py’

idlmam.py           100%[===================>]  21.55K  --.-KB/s    in 0s      

2026-03-29 21:08:27 (100 MB/s) - ‘idlmam.py’ saved [22064/22064]



In [2]:
import sys
sys.path.append('/home/jj/PyProjects/DeepLearning/Data/DATA')
sys.path.append('/idlmam.py')

In [3]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
from torchvision import transforms

from torch.utils.data import Dataset, DataLoader

from tqdm.autonotebook import tqdm

import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from matplotlib.pyplot import imshow

import pandas as pd

from sklearn.metrics import accuracy_score

from idlmam import train_simple_network, Flatten, weight_reset

/tmp/ipykernel_895773/2919734960.py:9: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm
/home/jj/PyProjects/DSC428/idlmam.py:215: SyntaxWarning: invalid escape sequence '\e'
  """Train simple neural networks


In [4]:
zip_file_url = "https://download.pytorch.org/tutorial/data.zip"

import requests, zipfile, io
r = requests.get(zip_file_url)
z = zipfile.ZipFile(io.BytesIO(r.content))
z.extractall()


#We will use some code to remove UNICODE tokens to make life easy for us processing wise
#e.g., convert something like "Ślusàrski" to Slusarski

In [5]:
namge_language_data = {}
import unicodedata
import string

all_letters = string.ascii_letters + " .,;'"
n_letters = len(all_letters)
alphabet = {}
for i in range(n_letters):
    alphabet[all_letters[i]] = i

# Turn a Unicode string to plain ASCII (ref: https://stackoverflow.com/a/518232/2809427)
def unicodeToAscii(s):
    return ''.join(
        c for c in unicodedata.normalize('NFD', s)
        if unicodedata.category(c) != 'Mn'
        and c in all_letters
    )

#Loop through every language, open the zip file entry, and read all the lines from the text file.
for zip_path in z.namelist():
    if "data/names/" in zip_path and zip_path.endswith(".txt"):
        lang = zip_path[len("data/names/"):-len(".txt")]
        with z.open(zip_path) as myfile:
            lang_names = [unicodeToAscii(line).lower() for line in str(myfile.read(), encoding='utf-8').strip().split("\n")]
            namge_language_data[lang] = lang_names
        print(lang, ": ", len(lang_names)) #Print out the name of each language too.

Arabic :  2000
Chinese :  268
Czech :  519
Dutch :  297
English :  3668
French :  277
German :  724
Greek :  203
Irish :  232
Italian :  709
Japanese :  991
Korean :  94
Polish :  139
Portuguese :  74
Russian :  9408
Scottish :  100
Spanish :  298
Vietnamese :  73


#creating Dataset subclass specific to thir problem. Please note that every sequence will have a label, i.e., language

In [6]:
class LanguageNameDataset(Dataset):

    def __init__(self, lang_name_dict, vocabulary):
        self.label_names = [x for x in lang_name_dict.keys()]
        self.data = []
        self.labels = []
        self.vocabulary = vocabulary
        for y, language in enumerate(self.label_names):
            for sample in lang_name_dict[language]:
                self.data.append(sample)
                self.labels.append(y)

    def __len__(self):
        return len(self.data)

    def string2InputVec(self, input_string):
        """
        This method will convert any input string into a vector of long values, according to the vocabulary used by this object.
        input_string: the string to convert to a tensor
        """
        T = len(input_string) #How many characters long is the string?

        #Create a new tensor to store the result in
        name_vec = torch.zeros((T), dtype=torch.long)
        #iterate through the string and place the appropriate values into the tensor
        for pos, character in enumerate(input_string):
            name_vec[pos] = self.vocabulary[character]

        return name_vec

    def __getitem__(self, idx):
        name = self.data[idx]
        label = self.labels[idx]

        #Convert the correct class label into a tensor for PyTorch
        #label_vec = torch.tensor([label], dtype=torch.long)

        return self.string2InputVec(name), label

In [17]:
dataset = LanguageNameDataset(namge_language_data, alphabet)

train_data, test_data = torch.utils.data.random_split(dataset, (len(dataset)-300, 300))
train_loader = DataLoader(train_data, batch_size=3, shuffle=True)
test_loader = DataLoader(test_data, batch_size=3, shuffle=False)

In [8]:
with torch.no_grad():
    input_sequence = torch.tensor([0, 1, 1, 0, 2], dtype=torch.long)
    embd = nn.Embedding(3, 2)
    x_seq = embd(input_sequence)
    print(input_sequence.shape, x_seq.shape)
    print(x_seq)

torch.Size([5]) torch.Size([5, 2])
tensor([[ 1.1727, -1.5234],
        [-1.4749, -0.2275],
        [-1.4749, -0.2275],
        [ 1.1727, -1.5234],
        [-1.3348,  0.1557]])


In [9]:
class LastTimeStep(nn.Module):
    """
    A class for extracting the hidden activations of the last time step following
    the output of a PyTorch RNN module.
    """
    def __init__(self, rnn_layers=1, bidirectional=False):
        super(LastTimeStep, self).__init__()
        self.rnn_layers = rnn_layers
        if bidirectional:
            self.num_directions = 2
        else:
            self.num_directions = 1

    def forward(self, input):
        #Result is either a tupe (out, h_t)
        #or a tuple (out, (h_t, c_t))
        rnn_output = input[0]
        last_step = input[1] #this will be h_t
        if(type(last_step) == tuple):#unless it's a tuple,
            last_step = last_step[0]#then h_t is the first item in the tuple
        batch_size = last_step.shape[1] #per docs, shape is: '(num_layers * num_directions, batch, hidden_size)'
        #reshaping so that everything is separate
        last_step = last_step.view(self.rnn_layers, self.num_directions, batch_size, -1) #(S, D, B, H)
        #print(last_step.shape)
        #We want the last layer's results
        last_step = last_step[self.rnn_layers-1]
        #print(last_step.shape)
        #Re order so batch comes first
        last_step = last_step.permute(1, 0, 2) # (D, B, H) --> (B, D, H)
        #print(last_step.shape)
        #print(last_step.reshape(batch_size, -1).shape)
        #Finally, flatten the last two dimensions into one
        return last_step.reshape(batch_size, -1) # (B, D * H)

In [10]:
#this part has the same code as the following code block but dimensions were confusin. So, I've corrected and added it in another code block
'''D = 64
vocab_size = len(all_letters)
hidden_nodes = 256
classes = len(dataset.label_names)

first_rnn = nn.Sequential(
  nn.Embedding(vocab_size, D), #(B, T) -> (B, T, D)
  nn.RNN(D, hidden_nodes, batch_first=True), #(B, T, D) -> ( (B,T,D) , (S, B, D)  )
  #the tanh activation is built into the RNN object, so we don't need to do it here
  LastTimeStep(), #We need to take the RNN output and reduce it to one item, (B, D)
  nn.Linear(hidden_nodes, classes), #(B, D) -> (B, classes)
)
'''

"D = 64\nvocab_size = len(all_letters)\nhidden_nodes = 256\nclasses = len(dataset.label_names)\n\nfirst_rnn = nn.Sequential(\n  nn.Embedding(vocab_size, D), #(B, T) -> (B, T, D)\n  nn.RNN(D, hidden_nodes, batch_first=True), #(B, T, D) -> ( (B,T,D) , (S, B, D)  )\n  #the tanh activation is built into the RNN object, so we don't need to do it here\n  LastTimeStep(), #We need to take the RNN output and reduce it to one item, (B, D)\n  nn.Linear(hidden_nodes, classes), #(B, D) -> (B, classes)\n)\n"

In [18]:
D = 64
vocab_size = len(all_letters)
hidden_nodes = 256
classes = len(dataset.label_names)

first_rnn = nn.Sequential(
  nn.Embedding(vocab_size, D),                  # (B, T) -> (B, T, D)
  nn.RNN(D, hidden_nodes, batch_first=True),    # (B, T, D) -> ((B, T, H), (S, B, H))
  # tanh is built into nn.RNN
  LastTimeStep(),                               # ((B, T, H), (S, B, H)) -> (B, H)
  nn.Linear(hidden_nodes, classes),             # (B, H) -> (B, classes)
)

In [19]:
device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")

In [20]:
loss_func = nn.CrossEntropyLoss()
batch_one_train = train_simple_network(first_rnn, loss_func, train_loader, test_loader=test_loader, score_funcs={'Accuracy': accuracy_score}, device=device, epochs=1)

Epoch:   0%|          | 0/1 [00:00<?, ?it/s]

Training:   0%|          | 0/6592 [00:00<?, ?it/s]

/home/jj/PyProjects/DSC428/idlmam.py:215: SyntaxWarning: invalid escape sequence '\e'
  """Train simple neural networks


RuntimeError: stack expects each tensor to be equal size, but got [7] at entry 0 and [9] at entry 1

In [21]:
'''
Just in case this saves weird here is the error copy/pasted

/home/jj/PyProjects/DSC428/idlmam.py:215: SyntaxWarning: invalid escape sequence '\e'
  """Train simple neural networks
---------------------------------------------------------------------------
RuntimeError                              Traceback (most recent call last)
Cell In[20], line 2
      1 loss_func = nn.CrossEntropyLoss()
----> 2 batch_one_train = train_simple_network(first_rnn, loss_func, train_loader, test_loader=test_loader, score_funcs={'Accuracy': accuracy_score}, device=device, epochs=1)

File ~/PyProjects/DSC428/idlmam.py:129, in train_simple_network(model, loss_func, train_loader, test_loader, score_funcs, epochs, device, checkpoint_file, lr)
    126 for epoch in tqdm(range(epochs), desc="Epoch"):
    127     model = model.train()#Put our model in training mode
--> 129     total_train_time += run_epoch(model, optimizer, train_loader, loss_func, device, results, score_funcs, prefix="train", desc="Training")
    131     results["total time"].append( total_train_time )
    132     results["epoch"].append( epoch )

File ~/PyProjects/DSC428/idlmam.py:54, in run_epoch(model, optimizer, data_loader, loss_func, device, results, score_funcs, prefix, desc)
     52 y_pred = []
     53 start = time.time()
---> 54 for inputs, labels in tqdm(data_loader, desc=desc, leave=False):
     55     #Move the batch to the device we are using. 
     56     inputs = moveTo(inputs, device)
     57     labels = moveTo(labels, device)

File ~/anaconda3/lib/python3.12/site-packages/tqdm/notebook.py:250, in tqdm_notebook.__iter__(self)
    248 try:
    249     it = super().__iter__()
--> 250     for obj in it:
    251         # return super(tqdm...) will not catch exception
    252         yield obj
    253 # NB: except ... [ as ...] breaks IPython async KeyboardInterrupt

File ~/anaconda3/lib/python3.12/site-packages/tqdm/std.py:1181, in tqdm.__iter__(self)
   1178 time = self._time
   1180 try:
-> 1181     for obj in iterable:
   1182         yield obj
   1183         # Update and possibly print the progressbar.
   1184         # Note: does not call self.update(1) for speed optimisation.

File ~/anaconda3/lib/python3.12/site-packages/torch/utils/data/dataloader.py:741, in _BaseDataLoaderIter.__next__(self)
    738 if self._sampler_iter is None:
    739     # TODO(https://github.com/pytorch/pytorch/issues/76750)
    740     self._reset()  # type: ignore[call-arg]
--> 741 data = self._next_data()
    742 self._num_yielded += 1
    743 if (
    744     self._dataset_kind == _DatasetKind.Iterable
    745     and self._IterableDataset_len_called is not None
    746     and self._num_yielded > self._IterableDataset_len_called
    747 ):

File ~/anaconda3/lib/python3.12/site-packages/torch/utils/data/dataloader.py:801, in _SingleProcessDataLoaderIter._next_data(self)
    799 def _next_data(self):
    800     index = self._next_index()  # may raise StopIteration
--> 801     data = self._dataset_fetcher.fetch(index)  # may raise StopIteration
    802     if self._pin_memory:
    803         data = _utils.pin_memory.pin_memory(data, self._pin_memory_device)

File ~/anaconda3/lib/python3.12/site-packages/torch/utils/data/_utils/fetch.py:57, in _MapDatasetFetcher.fetch(self, possibly_batched_index)
     55 else:
     56     data = self.dataset[possibly_batched_index]
---> 57 return self.collate_fn(data)

File ~/anaconda3/lib/python3.12/site-packages/torch/utils/data/_utils/collate.py:401, in default_collate(batch)
    340 def default_collate(batch):
    341     r"""
    342     Take in a batch of data and put the elements within the batch into a tensor with an additional outer dimension - batch size.
    343 
   (...)
    399         >>> default_collate(batch)  # Handle `CustomType` automatically
    400     """
--> 401     return collate(batch, collate_fn_map=default_collate_fn_map)

File ~/anaconda3/lib/python3.12/site-packages/torch/utils/data/_utils/collate.py:215, in collate(batch, collate_fn_map)
    209 transposed = list(
    210     zip(*batch, strict=False)
    211 )  # It may be accessed twice, so we use a list.
    213 if isinstance(elem, tuple):
    214     return [
--> 215         collate(samples, collate_fn_map=collate_fn_map)
    216         for samples in transposed
    217     ]  # Backwards compatibility.
    218 else:
    219     try:

File ~/anaconda3/lib/python3.12/site-packages/torch/utils/data/_utils/collate.py:155, in collate(batch, collate_fn_map)
    153 if collate_fn_map is not None:
    154     if elem_type in collate_fn_map:
--> 155         return collate_fn_map[elem_type](batch, collate_fn_map=collate_fn_map)
    157     for collate_type in collate_fn_map:
    158         if isinstance(elem, collate_type):

File ~/anaconda3/lib/python3.12/site-packages/torch/utils/data/_utils/collate.py:275, in collate_tensor_fn(batch, collate_fn_map)
    273     storage = elem._typed_storage()._new_shared(numel, device=elem.device)
    274     out = elem.new(storage).resize_(len(batch), *list(elem.size()))
--> 275 return torch.stack(batch, 0, out=out)

RuntimeError: stack expects each tensor to be equal size, but got [7] at entry 0 and [9] at entry 1'''


<>:1: SyntaxWarning: invalid escape sequence '\e'
<>:1: SyntaxWarning: invalid escape sequence '\e'
/tmp/ipykernel_895773/1158182795.py:1: SyntaxWarning: invalid escape sequence '\e'
  '''


'\nJust in case this saves weird here is the error copy/pasted\n\n/home/jj/PyProjects/DSC428/idlmam.py:215: SyntaxWarning: invalid escape sequence \'\\e\'\n  """Train simple neural networks\n---------------------------------------------------------------------------\nRuntimeError                              Traceback (most recent call last)\nCell In[20], line 2\n      1 loss_func = nn.CrossEntropyLoss()\n----> 2 batch_one_train = train_simple_network(first_rnn, loss_func, train_loader, test_loader=test_loader, score_funcs={\'Accuracy\': accuracy_score}, device=device, epochs=1)\n\nFile ~/PyProjects/DSC428/idlmam.py:129, in train_simple_network(model, loss_func, train_loader, test_loader, score_funcs, epochs, device, checkpoint_file, lr)\n    126 for epoch in tqdm(range(epochs), desc="Epoch"):\n    127     model = model.train()#Put our model in training mode\n--> 129     total_train_time += run_epoch(model, optimizer, train_loader, loss_func, device, results, score_funcs, prefix="train

#please change the batch size and try to run your code. You are expected to get an error. Please submit your notebook with that error printed.